# OmniObject3D House Subset — Dataset Walkthrough & Results

This notebook does four things, in order:

1. **Walks** the on-disk dataset (`dataset/`) and reports what is there.
2. **Inspects** it for integrity/failure points (missing renders, bad point clouds, class imbalance).
3. **Suggests** cleaning / imputation actions you can act on.
4. **Ingests** whatever model results exist under `results/` and renders comparison tables/figures.

It is defensive: if data or results are missing, each section says so instead of crashing.

See `docs/ROADMAP.md` for the pipeline and where artifacts belong.

In [ ]:
import json
import os
import re
import sys
from pathlib import Path

import numpy as np

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    import pandas as pd
except ImportError:
    pd = None

ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "dataset").is_dir() or (ROOT / "docs" / "ROADMAP.md").is_file():
        break
    ROOT = ROOT.parent
DATASET = ROOT / "dataset"
RESULTS = ROOT / "results"
FIGDIR = RESULTS / "comparison" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

PIX2VOX_TAX = ROOT / "Pix2Vox" / "datasets" / "OmniObject3D" / "OmniObject3D.json"
ATLASNET_TAX = ROOT / "AtlasNet" / "dataset" / "data" / "taxonomy.json"

print("project root :", ROOT)
print("dataset dir  :", DATASET, "| exists:", DATASET.is_dir())
print("results dir  :", RESULTS, "| exists:", RESULTS.is_dir())

## 1. Discovery — what categories and objects exist?

Categories are discovered from disk (never hardcoded) so this works whether you downloaded
the 18 bedroom, the 32 extras, or the full 50.

In [ ]:
def disk_categories(dataset: Path):
    found = set()
    for sub in ("point_clouds", "renders"):
        root = dataset / sub
        if root.is_dir():
            found |= {p.name for p in root.iterdir() if p.is_dir()}
    return sorted(found)


CATEGORIES = disk_categories(DATASET)
print(f"{len(CATEGORIES)} categories on disk")
print(" ".join(CATEGORIES) if CATEGORIES else "(none — run the download scripts first)")

## 2. Dataset information

Per-category object counts, image counts, and a long-tail view. Also parses the prepared
taxonomy/split files if `prepare_model_data.py` has been run.

In [ ]:
def count_pngs(obj_dir: Path) -> int:
    return len(list(obj_dir.glob("*.png")))


rows = []
for cat in CATEGORIES:
    pc_dir = DATASET / "point_clouds" / cat
    rd_dir = DATASET / "renders" / cat
    pcs = sorted(p.stem for p in pc_dir.glob("*.npy")) if pc_dir.is_dir() else []
    objs = sorted(p.name for p in rd_dir.iterdir() if p.is_dir()) if rd_dir.is_dir() else []
    imgs = sum(count_pngs(rd_dir / o) for o in objs) if rd_dir.is_dir() else 0
    rows.append({
        "category": cat,
        "objects_pc": len(pcs),
        "objects_render": len(objs),
        "total_pngs": imgs,
        "avg_views": round(imgs / len(objs), 1) if objs else 0.0,
    })

df = pd.DataFrame(rows) if pd is not None else None
if df is not None:
    df = df.sort_values("objects_pc", ascending=False).reset_index(drop=True)
    total_obj = int(df["objects_pc"].sum())
    print(f"TOTAL: {total_obj} objects, {int(df['total_pngs'].sum())} images, "
          f"{len(df)} categories")
    print(f"long-tail: max={df['objects_pc'].max()} ({df.iloc[0]['category']}), "
          f"min={df['objects_pc'].min()} ({df.iloc[-1]['category']}), "
          f"median={df['objects_pc'].median():.1f}")
    display(df)
else:
    print("pandas not installed — install with `pip install pandas` for tables.")
    for r in sorted(rows, key=lambda r: -r["objects_pc"]):
        print(r)

In [ ]:
if df is not None and len(df):
    fig, ax = plt.subplots(figsize=(max(8, len(df) * 0.32), 4.5))
    ax.bar(df["category"], df["objects_pc"], color="#4C72B0")
    ax.set_title("Objects per category (long-tail distribution)")
    ax.set_ylabel("# objects")
    ax.tick_params(axis="x", rotation=90, labelsize=7)
    fig.tight_layout()
    out = FIGDIR / "objects_per_category.png"
    fig.savefig(out, dpi=150)
    print("saved", out.relative_to(ROOT))
    plt.show()
else:
    print("no data to plot")

In [ ]:
def read_taxonomy(path: Path, id_key: str):
    if not path.is_file():
        return None
    try:
        data = json.loads(path.read_text())
    except Exception as exc:
        print(f"could not parse {path.name}: {exc}")
        return None
    return data


p2v = read_taxonomy(PIX2VOX_TAX, "taxonomy_id")
atl = read_taxonomy(ATLASNET_TAX, "synsetId")

if p2v:
    split_rows = []
    for entry in p2v:
        split_rows.append({
            "category": entry.get("taxonomy_id", "?"),
            "train": len(entry.get("train", [])),
            "val": len(entry.get("val", [])),
            "test": len(entry.get("test", [])),
        })
    sdf = pd.DataFrame(split_rows) if pd is not None else None
    if sdf is not None:
        sdf["total"] = sdf[["train", "val", "test"]].sum(axis=1)
        print(f"Pix2Vox taxonomy: {len(sdf)} categories, "
              f"train={sdf['train'].sum()} val={sdf['val'].sum()} test={sdf['test'].sum()}")
        display(sdf)
else:
    print(f"no Pix2Vox taxonomy at {PIX2VOX_TAX.relative_to(ROOT)} "
          "(run prepare_model_data.py)")

print("AtlasNet taxonomy:", len(atl), "categories" if atl else "(missing)")

## 3. Integrity checks & failure points

We look for the things that actually break training or corrupt metrics.

In [ ]:
EXPECTED_VIEWS = 24
issues = []

for cat in CATEGORIES:
    pc_dir = DATASET / "point_clouds" / cat
    rd_dir = DATASET / "renders" / cat

    pcs = {p.stem for p in pc_dir.glob("*.npy")} if pc_dir.is_dir() else set()
    objs = {p.name for p in rd_dir.iterdir() if p.is_dir()} if rd_dir.is_dir() else set()

    for obj in sorted(pcs - objs):
        issues.append((cat, obj, "point cloud without renders"))
    for obj in sorted(objs - pcs):
        issues.append((cat, obj, "renders without point cloud"))

    for obj in sorted(objs):
        o = rd_dir / obj
        n = count_pngs(o)
        if n < EXPECTED_VIEWS:
            issues.append((cat, obj, f"only {n}/{EXPECTED_VIEWS} renders"))
        if not (o / "transforms.json").is_file():
            issues.append((cat, obj, "missing transforms.json"))

print(f"{len(issues)} structural issue(s)")
for cat, obj, msg in issues[:40]:
    print(f"  {cat:16s} {obj:20s} {msg}")
if len(issues) > 40:
    print(f"  ... and {len(issues) - 40} more")

In [ ]:
pc_rows = []
for cat in CATEGORIES:
    pc_dir = DATASET / "point_clouds" / cat
    if not pc_dir.is_dir():
        continue
    for f in sorted(pc_dir.glob("*.npy")):
        rec = {"category": cat, "object": f.stem, "n_points": None,
               "nan": None, "inf": None, "flat": None, "dup_frac": None,
               "aspect": None, "error": None}
        try:
            pts = np.load(f).astype(np.float64)
            if pts.ndim != 2 or pts.shape[1] != 3:
                rec["error"] = f"bad shape {pts.shape} (expected (N,3))"
                pc_rows.append(rec)
                continue
            rec["n_points"] = int(pts.shape[0])
            rec["nan"] = int(np.isnan(pts).any())
            rec["inf"] = int(np.isinf(pts).any())
            if pts.size and np.isfinite(pts).all():
                span = pts.max(0) - pts.min(0)
                rec["flat"] = int((span < 1e-6).any())
                nz = span[span > 1e-8]
                rec["aspect"] = round(float(nz.max() / nz.min()), 2) if len(nz) else 0.0
                uniq = np.unique(np.round(pts, 6), axis=0).shape[0]
                rec["dup_frac"] = round(1.0 - uniq / pts.shape[0], 3)
        except Exception as exc:
            rec["error"] = str(exc)
        pc_rows.append(rec)

pdf = pd.DataFrame(pc_rows) if pd is not None else None
problems = [r for r in pc_rows if r["error"] or r["nan"] or r["inf"] or r["flat"]]
print(f"scanned {len(pc_rows)} point clouds | {len(problems)} problematic")
for r in problems[:40]:
    print("  ", r)
if pdf is not None and len(pdf):
    ok = pdf.dropna(subset=["n_points"])
    if len(ok):
        print("\npoints per cloud: min=%d max=%d median=%.0f" % (
            ok["n_points"].min(), ok["n_points"].max(), ok["n_points"].median()))
        print("aspect ratio (elongation) p95=%.1f  max=%.1f" % (
            ok["aspect"].quantile(0.95), ok["aspect"].max()))
        print("duplicate-point fraction: p95=%.3f max=%.3f" % (
            ok["dup_frac"].quantile(0.95), ok["dup_frac"].max()))

In [ ]:
if df is not None and len(df):
    small = df[df["objects_pc"] < 3]
    print(f"categories with <3 objects (not reliably trainable): {len(small)}")
    if len(small):
        print(small[["category", "objects_pc"]].to_string(index=False))
    tot = df["objects_pc"].sum()
    if tot:
        top20 = df["objects_pc"].iloc[: max(1, len(df) // 5)].sum() / tot
        print(f"\ntop ~20% of categories hold {top20:.0%} of all objects")

## 4. Cleaning / imputation suggestions

Generated from the checks above. These are recommendations, not automatic edits —
review before applying.

In [ ]:
suggestions = []

bad_struct = [i for i in issues if "renders without point cloud" in i[2]
              or "point cloud without renders" in i[2]]
if bad_struct:
    suggestions.append(
        f"{len(bad_struct)} object(s) exist in only one modality. Re-download the "
        "missing modality, or exclude the object from both layouts and re-run "
        "prepare_model_data.py.")
short = [i for i in issues if "renders" in i[2] and "/24" in i[2]]
if short:
    suggestions.append(
        f"{len(short)} object(s) have <24 views. Either drop them or use the views "
        "available (the loader tolerates <24; metrics on them are weaker).")

notrans = [i for i in issues if "transforms.json" in i[2]]
if notrans:
    suggestions.append(
        f"{len(notrans)} object(s) miss transforms.json; needed only for pose-aware "
        "metrics, safe to ignore for image->voxel/mesh training.")

bad_pc = [r for r in pc_rows if r["error"]]
if bad_pc:
    suggestions.append(
        f"{len(bad_pc)} unreadable/incorrectly shaped point clouds. Re-download those "
        "categories (they will corrupt voxelization/fusion).")

nan_pc = [r for r in pc_rows if r["nan"] or r["inf"]]
if nan_pc:
    suggestions.append(
        f"{len(nan_pc)} clouds contain NaN/Inf. Prefer re-download; otherwise drop "
        "non-finite points and re-normalize (fallback imputation).")

flat_pc = [r for r in pc_rows if r["flat"]]
if flat_pc:
    suggestions.append(
        f"{len(flat_pc)} clouds are degenerate (near-zero extent on an axis). Voxelization "
        "is unstable; exclude them.")

if df is not None and len(df):
    tot = int(df["objects_pc"].sum())
    small = df[df["objects_pc"] < 3]["category"].tolist()
    if small:
        suggestions.append(
            f"{len(small)} categories have <3 objects ({', '.join(small[:8])}"
            f"{'...' if len(small) > 8 else ''}). Report them separately; do not claim "
            "generalization from them.")
    if tot and df["objects_pc"].max() > 4 * max(1, df["objects_pc"].median()):
        suggestions.append(
            "Distribution is long-tailed. Options: cap each class at N objects for "
            "balanced training, or oversample rare classes; report full-data numbers "
            "as the headline.")

if not suggestions:
    print("No issues detected — dataset looks clean for this pipeline.")
else:
    print("Recommended actions:")
    for i, s in enumerate(suggestions, 1):
        print(f"  {i}. {s}")

## 5. Results ingestion

Reads `results/**/summary.json` and CSVs produced by training runs and renders
comparison tables. See `docs/ROADMAP.md` §4 for the expected layout.

If you have a raw Pix2Vox `TEST RESULTS` log, the **next cell** parses/persists it.

In [ ]:
def parse_pix2vox_test_table(text: str):
    """Parse Pix2Vox 'TEST RESULTS' tab/space table -> (rows, overall)."""
    lines = text.splitlines()
    start = None
    for i, ln in enumerate(lines):
        if "TEST RESULTS" in ln:
            start = i
            break
    if start is None:
        return None, None
    rows, overall = [], None
    for ln in lines[start + 1:]:
        s = ln.strip()
        if not s:
            if rows:
                break
            continue
        if s.startswith("Taxonomy") or set(s) <= {"=", "-", " "}:
            continue
        parts = re.split(r"\s+", s)
        if parts[0].lower().startswith("overall"):
            vals = [p for p in parts[1:] if re.match(r"^-?\d*\.?\d+$", p)]
            overall = [float(v) for v in vals]
            continue
        if len(parts) >= 2 and parts[0] not in ("Taxonomy",):
            name = parts[0]
            nums = [p for p in parts[1:] if re.match(r"^-?\d*\.?\d+$", p)]
            if nums:
                try:
                    n = int(float(nums[0]))
                except ValueError:
                    n = None
                iou = [float(x) for x in nums[1:]]
                rows.append({"category": name, "n_sample": n, "iou": iou})
    return rows, overall


print("parser ready")

In [ ]:
# If you saved a raw Pix2Vox run, drop its TEST RESULTS text here to ingest it.
RAW_PIX2VOX_LOG = (RESULTS / "pix2vox" / "manual_test_results.txt")
if RAW_PIX2VOX_LOG.is_file():
    rows_iou, overall_iou = parse_pix2vox_test_table(RAW_PIX2VOX_LOG.read_text())
    if rows_iou:
        THRESH = [0.20, 0.30, 0.40, 0.50]
        recs = []
        for r in rows_iou:
            d = {"category": r["category"], "n_sample": r["n_sample"]}
            for t, v in zip(THRESH, r["iou"]):
                d[f"t={t:.2f}"] = v
            recs.append(d)
        df_iou = pd.DataFrame(recs) if pd is not None else None
        if df_iou is not None:
            display(df_iou)
            print("overall:", overall_iou)
        else:
            print(rows_iou, "overall:", overall_iou)
    else:
        print("no parseable table found")
else:
    print(f"no manual Pix2Vox log at {RAW_PIX2VOX_LOG.relative_to(ROOT)}")
    print("note: the committed Pix2Vox stdout covers this table")
    print("      (results/pix2vox/<run-id>/stdout.log)")

In [ ]:
def load_summaries(base: Path):
    out = []
    if not base.is_dir():
        return out
    for s in sorted(base.glob("*/summary.json")):
        try:
            d = json.loads(s.read_text())
            d["_run"] = s.parent.name
            out.append(d)
        except Exception as exc:
            print(f"skip {s}: {exc}")
    return out


p2v_runs = load_summaries(RESULTS / "pix2vox")
atl_runs = load_summaries(RESULTS / "atlasnet")
print(f"Pix2Vox runs: {len(p2v_runs)} | AtlasNet runs: {len(atl_runs)}")

if p2v_runs:
    print("\nPix2Vox summaries:")
    for r in p2v_runs:
        print(" ", r.get("_run"), "|", {k: v for k, v in r.items() if k != "_run"})
if atl_runs:
    print("\nAtlasNet summaries:")
    for r in atl_runs:
        print(" ", r.get("_run"), "|", {k: v for k, v in r.items() if k != "_run"})
if not p2v_runs and not atl_runs:
    print("No runs found yet — train, then place summaries under results/ per ROADMAP.")

In [ ]:
comp_rows = []
for r in p2v_runs:
    comp_rows.append({
        "model": "Pix2Vox",
        "run": r.get("_run"),
        "task": "image->voxel",
        "headline": r.get("best_iou"),
        "metric": f"IoU@t={r.get('best_t')}",
    })
for r in atl_runs:
    comp_rows.append({
        "model": "AtlasNet-SVR",
        "run": r.get("_run"),
        "task": "image->surface",
        "headline": r.get("best_fscore", r.get("best_chamfer")),
        "metric": "F-score" if r.get("best_fscore") is not None else "Chamfer",
    })

cdf = pd.DataFrame(comp_rows) if pd is not None else None
if cdf is not None and len(cdf):
    cpath = RESULTS / "comparison" / "metrics_table.csv"
    cdf.to_csv(cpath, index=False)
    print("wrote", cpath.relative_to(ROOT))
    display(cdf)
else:
    print("no comparison table yet")

## 6. Write the results index (`results/README.md`)

Keeps a human-readable record of every run and the generated figures.

In [ ]:
lines = ["# Results index", "",
         "Auto-generated by `notebooks/dataset_analysis.ipynb`. Do not edit by hand.", "",
         "## Dataset", ""]
if df is not None and len(df):
    lines += [f"- categories: {len(df)}",
              f"- objects: {int(df['objects_pc'].sum())}",
              f"- images: {int(df['total_pngs'].sum())}",
              f"- structural issues: {len(issues)}", ""]
else:
    lines += ["- dataset not found", ""]

lines += ["## Runs", ""]
run_lines = [f"- **{r['model']}** `{r.get('run')}` — {r['metric']} = {r.get('headline')}"
             for r in comp_rows]
lines += run_lines if run_lines else ["- (none yet)"]
lines += ["", "## Figures", ""]
figs = sorted(p.name for p in FIGDIR.glob("*.png"))
lines += [f"- `results/comparison/figures/{f}`" for f in figs] if figs else ["- (none yet)"]

out_md = RESULTS / "README.md"
out_md.write_text("\n".join(lines) + "\n")
print("wrote", out_md.relative_to(ROOT))
print("\n".join(lines))

## 7. Summary

Run of this notebook produced the dataset report, the failure-point list, the
cleaning recommendations, and the results index. Re-run after every training
run or dataset change.